# 02 — Real VLM extraction and deterministic audit

รัน Qwen VLM จริงบนภาพ synthetic/public แล้วส่งผลผ่าน normalization, rules และ decision policy ใน package. Expected runtime ขึ้นกับ GPU/Colab quota และ model download ครั้งแรกอาจใช้เวลาหลายนาที

> Privacy: ห้ามใช้เอกสารลูกค้าจริงหรือ PII. Notebook ไม่เปิด Gradio/public tunnel และ output default เป็น redacted public artifact.

## 1. Idempotent standalone bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get('INVOICE_AUDITOR_REPO', 'https://github.com/Sayomphon/Multimodal_Invoice_Auditor.git')
APP_REF = os.environ.get('INVOICE_AUDITOR_REF', 'main')
PROJECT_ROOT = Path('/content/Multimodal_Invoice_Auditor')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
git_env = os.environ.copy()
if GITHUB_TOKEN:
    git_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'Authorization: Bearer {GITHUB_TOKEN}'})
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(PROJECT_ROOT)], check=True, env=git_env)
if not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'Existing path is not a Git checkout: {PROJECT_ROOT}')
if subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Checkout contains local changes; use a clean runtime')
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth=1', 'origin', APP_REF], check=True, env=git_env)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
APP_COMMIT = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-colab.lock'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', '-e', '.'], check=True)
print({'project_root': str(PROJECT_ROOT), 'application_commit': APP_COMMIT})

## 2. Runtime preflight and immutable model registry

In [ ]:
from invoice_auditor.runtime import detect_runtime
from invoice_auditor.vlm_runtime import default_registry_path, load_model_registry

environment = detect_runtime(disk_path=PROJECT_ROOT)
if not environment.cuda_available:
    raise RuntimeError('CUDA GPU unavailable. Select Runtime > Change runtime type > GPU.')
registry = load_model_registry(default_registry_path())
print(environment.model_dump(mode='json'))
print(registry.model_dump(mode='json'))
if registry.acceptance_status != 'colab_verified':
    print('CANDIDATE MODE: pins/revisions require one-image smoke, freeze, and clean-runtime rerun.')

## 3. Generate a fixed synthetic golden set
Colab may need a Thai font package once. This affects synthetic rendering only, not invoice business logic.

In [ ]:
from invoice_auditor.synthetic_generator import generate_dataset, resolve_thai_font

try:
    thai_font = resolve_thai_font()
except FileNotFoundError:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-noto-core'], check=True)
    thai_font = resolve_thai_font()
DATASET_ROOT = PROJECT_ROOT / 'data' / 'synthetic'
MANIFEST_PATH = generate_dataset(DATASET_ROOT, count=6, seed=42, font_path=thai_font, overwrite=True)
print(MANIFEST_PATH)

## 4. Optional Colab upload
Default acceptance uses the synthetic manifest. Set `USE_UPLOAD=True` only for synthetic/public image or one-page PDF.

In [ ]:
USE_UPLOAD = False
if USE_UPLOAD:
    import json
    from google.colab import files
    from invoice_auditor.io_utils import atomic_write_json
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one image or one-page PDF')
    original_name, content = next(iter(uploaded.items()))
    safe_name = Path(original_name).name
    upload_root = PROJECT_ROOT / 'data' / 'synthetic' / 'uploads'
    upload_root.mkdir(parents=True, exist_ok=True)
    upload_path = upload_root / safe_name
    upload_path.write_bytes(content)
    MANIFEST_PATH = atomic_write_json(upload_root / 'manifest.json', {
        'schema_version': '1.0.0',
        'records': [{'image_id': 'colab-upload-001', 'image_path': safe_name}],
    })
print({'manifest': str(MANIFEST_PATH), 'upload_enabled': USE_UPLOAD})

## 5. Load primary/fallback runtime once and audit one image

In [ ]:
import json
from IPython.display import JSON, display
from invoice_auditor.pipeline import InvoiceAuditPipeline
from invoice_auditor.presentation import build_public_audit_view
from invoice_auditor.vlm_runtime import VLMRuntime

model_runtime = VLMRuntime(registry, allow_download=True, runtime_info=environment)
pipeline = InvoiceAuditPipeline()
manifest = json.loads(Path(MANIFEST_PATH).read_text(encoding='utf-8'))
first_entry = manifest['records'][0]
first_path = Path(MANIFEST_PATH).parent / first_entry['image_path']
raw, trace = model_runtime.extract(first_path)
one_report = pipeline.audit(raw, source_id=first_entry['image_id'], model_trace=trace, register_duplicate=False)
display(JSON(build_public_audit_view(one_report)))

## 6. Batch inference, metrics, and acceptance artifacts

In [ ]:
import hashlib
from datetime import UTC, datetime
from invoice_auditor.artifacts import validate_run_artifacts
from invoice_auditor.batch_inference import run_batch_inference
from invoice_auditor.evaluation import evaluate_jsonl
from invoice_auditor.io_utils import atomic_write_json

RUN_ID = f"colab-{datetime.now(UTC):%Y%m%dT%H%M%SZ}"
RUN_DIR = PROJECT_ROOT / 'reports' / 'colab' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
atomic_write_json(RUN_DIR / 'environment.json', environment.model_dump(mode='json'))
batch = run_batch_inference(MANIFEST_PATH, RUN_DIR / 'predictions.jsonl', model_runtime, pipeline=pipeline, run_id=RUN_ID, public_output=True)
ground_truth = Path(MANIFEST_PATH).parent / 'ground_truth.jsonl'
if ground_truth.is_file():
    metrics = evaluate_jsonl(ground_truth, RUN_DIR / 'predictions.jsonl')
else:
    metrics = {'counts': {'attempted': batch.attempted}, 'notes': ['Upload mode has no ground truth.']}
atomic_write_json(RUN_DIR / 'metrics.json', metrics)
lock_sha = hashlib.sha256((PROJECT_ROOT / 'requirements-colab.lock').read_bytes()).hexdigest()
atomic_write_json(RUN_DIR / 'run_manifest.json', {
    'schema_version': '1.0.0',
    'run_id': RUN_ID,
    'application_commit': APP_COMMIT,
    'dependency_lock_sha256': lock_sha,
    'model_registry': registry.model_dump(mode='json'),
    'started_at': batch.started_at.isoformat(),
    'completed_at': batch.completed_at.isoformat(),
    'public_redacted': True,
})
validation = validate_run_artifacts(RUN_DIR, require_colab=True, minimum_attempts=(1 if USE_UPLOAD else 5))
display(JSON({'batch': batch.model_dump(mode='json'), 'metrics': metrics, 'validation': validation.model_dump(mode='json')}))
if not validation.ok:
    raise RuntimeError(f'Artifact validation failed: {validation.errors}')

## 7. Release GPU memory
รัน cell นี้เมื่อเสร็จ เพื่อคืน model references และ CUDA cache.

In [ ]:
model_runtime.release()
print('GPU model references released')